## Netflix Movie Recommendation System

In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
df=pd.read_csv("C:/Users/Apurva Malankar/Downloads/netflix_titles.csv/netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [7]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [10]:
df = df[['title','director','cast','listed_in','description']]
df=df.fillna('')

In [11]:
df['combined']=df['director'] + ' ' + df['cast'] + ' ' + df['listed_in'] + ' ' + df['description']

In [16]:
tfidf = TfidfVectorizer(stop_words='english')
matrix = tfidf.fit_transform(df['combined'])
similarity = cosine_similarity(matrix)

In [17]:
def recommend(title):

    index = df[df['title'] == title].index[0]
    scores = list(enumerate(similarity[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    for i in scores[1:6]:
        print(df.iloc[i[0]].title)

recommend("Narcos")

Wild District
El Cartel
El final del paraíso
La Viuda Negra
The Great Heist


In [18]:
recommend("Stranger Things")

Beyond Stranger Things
Prank Encounters
The Umbrella Academy
Anjaan: Special Crimes Unit
Eli


In [20]:
recommend("The Crown")

Downton Abbey
London Spy
Flowers
Lovesick
The Frankenstein Chronicles


## Solving Duplicate Titles Issue

In [24]:
df = df.reset_index(drop=True)
indices = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

The dataset contained multiple entries with identical titles, which could lead to incorrect indexing and inconsistent recommendation results.<br>
To handle duplicate content entries, a title indexing strategy was implemented by generating a lowercase title index mapping and removing duplicate keys.<br>
This technique improves recommendation accuracy and prevents ambiguity during similarity lookup.

## Recommending movies based on  cosine similarity scores.

In [31]:
def recommend(title, n=10):

    title = title.lower()

    if title not in indices:
        return "Movie not found"

    idx = indices[title]

    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:n+1]

    for i in sim_scores:
        movie = df.iloc[i[0]]['title']
        score = round(i[1]*100,2)
        print(movie, " → ", score,"%")

In [30]:
recommend("The Crown")

Downton Abbey  →  16.02 %
London Spy  →  11.1 %
Flowers  →  10.88 %
Lovesick  →  10.79 %
The Frankenstein Chronicles  →  10.39 %
Greatest Events of WWII in Colour  →  10.3 %
Stories by Rabindranath Tagore  →  10.18 %
The Nineties  →  9.83 %
The Exception  →  9.69 %
The Queen  →  9.68 %


This output displays the top recommended Netflix titles based on cosine similarity scores. <br>Higher similarity percentages indicate stronger content similarity with the selected title.